# How much do the experts agree with each other?

The quality benchmark scores models against a grade a human wrote down. This notebook asks
what that grade is worth, in the two datasets that kept their readers apart:
[FQS](../docs/datasets/fqs.md), where three graders each gave a three-class verdict and six
doctors each gave a score out of 100, and [MSHF](../docs/datasets/mshf.md), where three
annotators each rated illumination, clarity and contrast and gave an overall verdict.

It matters for two reasons. **A model cannot be scored above the reference's own reliability**
— if two readers only agree with each other two times in three, a model agreeing with their
consensus three times in four is not obviously worse than a human. And **where readers
disagree is where a model's mistakes are worth forgiving**, which is the difference between a
model that fails on hard photographs and one that fails on easy ones.

Nothing here scores a model. It reads `.atlas_data/` for the readers and `results/quality/`
only in section 5, to put the two ceilings side by side.


In [ ]:
import csv
import itertools
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

STORE = Path('..').resolve() / '.atlas_data'
RESULTS = Path('..').resolve() / 'results' / 'quality'

READERS = {'fqs': ['level1', 'level2', 'level3'],
           'mshf': ['annotator1', 'annotator2', 'annotator3']}

def readings(slug: str) -> pd.DataFrame:
    """Every reader's opinion about every photograph, one row each."""
    return pd.read_csv(STORE / slug / 'labels.csv').assign(dataset=slug)

def spread(slug: str, field: str = 'quality') -> pd.DataFrame:
    """One row per photograph, one column per reader, plus the published consensus."""
    frame = readings(slug)
    frame = frame[frame['field'] == field]
    return frame.pivot(index='key', columns='reader', values='value')

WORTH = ['good', 'usable']

def worth_measuring(frame: pd.DataFrame) -> pd.DataFrame:
    """The same opinions, on the question the benchmark actually asks."""
    return frame.isin(WORTH).replace({True: 'worth measuring', False: 'not'})

opinions = {slug: spread(slug) for slug in READERS}
# FQS grades in three classes and MSHF in two, so the two are not comparable as
# published — and the benchmark asks a two-class question of both. Every table below
# is computed twice for FQS: as its graders wrote it, and collapsed.
binary = {slug: worth_measuring(frame) for slug, frame in opinions.items()}

for slug, frame in opinions.items():
    print(f'{slug}: {len(frame)} photographs, readers {sorted(frame.columns)}, '
          f'{frame[READERS[slug]].stack().nunique()} classes')



## 1. How often do they simply agree?

The share of photographs where every reader wrote the same word.

**Read it twice.** [FQS](../docs/datasets/fqs.md) grades in three classes and
[MSHF](../docs/datasets/mshf.md) in two, so their raw agreement is not comparable as published:
three graders arguing about whether a photograph is *good* or merely *usable* count as disagreeing,
and two readers choosing between two words have an easier job. **And the benchmark asks a two-class
question of both** — is this photograph worth measuring, where `good` and `usable` are both yes —
so the reliability that bounds a model's score is the reliability *on that question*, not on the
published one. Every table here is therefore computed both ways for FQS. The gap between them is
the share of the disagreement that is about the middle class and never reaches the benchmark.


In [ ]:
def agreement(frame: pd.DataFrame, readers: list[str]) -> dict[str, object]:
    votes = frame[readers].dropna()
    return {
        'photographs': len(votes),
        'classes the readers used': votes.stack().nunique(),
        'all three agree': (votes.nunique(axis=1) == 1).mean(),
        'one reader differs': (votes.nunique(axis=1) == 2).mean(),
        'all three differ': (votes.nunique(axis=1) == 3).mean(),
    }

summary = []
for slug, readers in READERS.items():
    summary.append({'dataset': slug, 'grades': 'as published',
                    **agreement(opinions[slug], readers)})
    summary.append({'dataset': slug, 'grades': 'worth measuring, or not',
                    **agreement(binary[slug], readers)})
pd.DataFrame(summary).set_index(['dataset', 'grades']).round(3)


## 2. Agreement beyond chance, reader against reader

Cohen's κ for every pair of readers, on both questions. This is **the ceiling**: no model measured
against these grades can be expected to agree with them better than their authors agree with each
other, and a model that does is matching a consensus rather than matching a person.

The row that bounds a model's benchmark score is the **worth measuring, or not** one, because that
is the question the benchmark asks.


In [ ]:
pairs = []
for slug, readers in READERS.items():
    for grades, source in (('as published', opinions), ('worth measuring, or not', binary)):
        frame = source[slug][readers].dropna()
        for a, b in itertools.combinations(readers, 2):
            pairs.append({
                'dataset': slug, 'grades': grades, 'pair': f'{a} vs {b}',
                'raw agreement': (frame[a] == frame[b]).mean(),
                'kappa': cohen_kappa_score(frame[a], frame[b]),
            })

between_readers = pd.DataFrame(pairs).set_index(['dataset', 'grades', 'pair'])
between_readers.round(3)


### 2.1 Is one reader the odd one out?

Each reader against the grade the dataset published as agreed. A reader far below the others
here is not necessarily wrong — they may be the strict one — but they are the reader whose
opinion the consensus most often overrode, and a model trained or tuned on that consensus
inherits whatever the majority believed.


In [ ]:
against_consensus = []
for slug, readers in READERS.items():
    for grades, source in (('as published', opinions), ('worth measuring, or not', binary)):
        frame = source[slug]
        if 'consensus' not in frame.columns:
            continue
        for reader in readers:
            pair = frame[[reader, 'consensus']].dropna()
            against_consensus.append({
                'dataset': slug, 'grades': grades, 'reader': reader, 'photographs': len(pair),
                'kappa vs consensus': cohen_kappa_score(pair[reader], pair['consensus']),
            })
pd.DataFrame(against_consensus).set_index(['dataset', 'grades', 'reader']).round(3)


### 2.2 Is the published consensus the majority?

It should be, and it is worth checking rather than assuming: a consensus that is *not* the
majority is a fact about how the dataset was assembled, and any model scored against it is
being scored against that decision.


In [ ]:
for slug, readers in READERS.items():
    frame = opinions[slug]
    if 'consensus' not in frame.columns:
        continue
    frame = frame[[*readers, 'consensus']].dropna()
    majority = frame[readers].mode(axis=1)[0]
    matches = (frame['consensus'] == majority)
    print(f'{slug}: consensus is the majority on {matches.mean():.3%} of {len(frame)} photographs')
    for key, row in frame[~matches].head(5).iterrows():
        print('   ', key, dict(row))
    missing = len(opinions[slug]) - len(frame)
    if missing:
        print(f'    and {missing} photographs carry no published consensus at all')


## 3. What do they disagree *about*?

[MSHF](../docs/datasets/mshf.md) asks its annotators for three component ratings as well as an
overall verdict, so it can say which aspect of a photograph is hardest to agree on. That is
worth knowing before trusting any single quality label: an aspect the readers themselves
cannot settle is one no model will be scored fairly on.


In [ ]:
components = []
for field in ('illumination', 'clarity', 'contrast', 'quality'):
    frame = spread('mshf', field)[READERS['mshf']].dropna()
    kappas = [cohen_kappa_score(frame[a], frame[b])
              for a, b in itertools.combinations(READERS['mshf'], 2)]
    components.append({
        'rated': field, 'photographs': len(frame),
        'all three agree': (frame.nunique(axis=1) == 1).mean(),
        'mean pairwise kappa': np.mean(kappas),
    })
pd.DataFrame(components).set_index('rated').round(3)


### 3.1 FQS: six doctors, one number each

[FQS](../docs/datasets/fqs.md) also publishes six doctors' scores out of 100 for every
photograph. The spread between them is a continuous measure of how hard a photograph is to
judge — and it should be widest exactly where the three-class graders disagreed, which is a
check that the two annotation layers are describing the same difficulty.


In [ ]:
mos = spread('fqs', 'mos').astype(float)
doctors = [column for column in mos.columns if column.startswith('doc')]
mos['spread'] = mos[doctors].std(axis=1)

graders = opinions['fqs'][READERS['fqs']].dropna()
mos['readers agreeing'] = 4 - graders.nunique(axis=1)  # 3 when unanimous, 1 when all differ

print(f"median spread between six doctors: {mos['spread'].median():.1f} points out of 100")
figure, axis = plt.subplots(figsize=(7, 4))
mos.boxplot(column='spread', by='readers agreeing', ax=axis, grid=False)
axis.set_xlabel('how many of the three graders wrote the same word')
axis.set_ylabel('standard deviation of the six doctors, points')
axis.set_title('where the graders split, the doctors spread')
figure.suptitle('')
figure.tight_layout()


## 4. Where the disagreement lives

The photographs the readers could not settle, as images. These are the ones a model is most
likely to be marked wrong on, and the ones where being marked wrong means least.


In [ ]:
from PIL import Image

PER_DATASET = 6
for slug, readers in READERS.items():
    frame = opinions[slug][readers].dropna()
    split = frame[frame.nunique(axis=1) > 1]
    print(f'{slug}: {len(split)} of {len(frame)} photographs the readers did not settle')
    if not len(split):
        continue
    figure, axes = plt.subplots(1, PER_DATASET, figsize=(2.3 * PER_DATASET, 2.8))
    for axis, (key, row) in zip(np.atleast_1d(axes), split.head(PER_DATASET).iterrows()):
        axis.imshow(Image.open(STORE / slug / '512' / 'images' / f'{key}.png'))
        axis.set_title(f"{key}\n{' / '.join(row.tolist())}", fontsize=7)
        axis.axis('off')
    figure.tight_layout()
    plt.show()


## 5. The ceiling: models against the consensus, readers against each other

The comparison this notebook exists for. On the left, how well each model agreed with the
grade the dataset published; on the right, how well the dataset's own readers agreed with one
another. A model at or above the reader band is not beating the experts — it is matching a
**consensus**, which is steadier than any individual because disagreements were voted away.
But a model well below that band has room that is unambiguously its own.

Both numbers are κ, so they are on the same scale and can be read against each other —
and both are computed on the **two-class** question, `good` and `usable` together
against `bad`, which is the one the benchmark asks and the only one MSHF can answer.



In [ ]:
models = []
for path in sorted(RESULTS.glob('*/*.json')):
    record = json.loads(path.read_text())
    if record['dataset'] in READERS:
        models.append({
            'dataset': record['dataset'], 'model': record['model'],
            'kappa': record['summary']['gradeable']['kappa'],
        })
models = pd.DataFrame(models)

figure, axes = plt.subplots(1, len(READERS), figsize=(6 * len(READERS), 4), squeeze=False)
for axis, slug in zip(axes[0], READERS):
    here = models[models['dataset'] == slug].sort_values('kappa')
    axis.barh(here['model'], here['kappa'], color='tab:blue', label='model vs the consensus')
    # The benchmark's question is the two-class one, so that is the band to compare against.
    band = between_readers.loc[(slug, 'worth measuring, or not'), 'kappa']
    axis.axvspan(band.min(), band.max(), color='tab:orange', alpha=0.25,
                 label='reader against reader')
    for value in band:
        axis.axvline(value, color='tab:orange', linewidth=1)
    axis.set_title(slug)
    axis.set_xlabel("Cohen's κ")
    axis.set_xlim(0, 1)
    axis.tick_params(axis='y', labelsize=7)
    axis.legend(fontsize=7, loc='lower right')
figure.tight_layout()



In [ ]:
summary = models.pivot(index='model', columns='dataset', values='kappa')
for slug in READERS:
    band = between_readers.loc[(slug, 'worth measuring, or not'), 'kappa']
    summary.loc['— readers, worst pair', slug] = band.min()
    summary.loc['— readers, best pair', slug] = band.max()
summary.round(3)



## 6. What this says about the benchmark

- **A grade is not a fact.** Where the readers of a dataset agree with each other less than a
  model agrees with their consensus, the benchmark is measuring how well the model reproduces
  a committee, not how well it judges a photograph.
- **The two datasets are not equally hard references.** Read every model's score on each of
  them against that dataset's own reader band rather than against the other dataset's.
- **Much of FQS's disagreement is about the middle class.** Collapsing `good` and
  `usable` — the question the benchmark asks — moves its graders from agreeing on 46% of
  photographs to 69%, and their pairwise κ from 0.34–0.51 to 0.49–0.63. The rest of the
  disagreement is about a distinction no score in this benchmark depends on.
- **Disagreement is a property of the photograph.** Section 3.1 shows the six doctors'
  spread widening exactly where the three graders split, which is what makes the split
  believable as a difficulty signal rather than as one grader being careless.
- **What would settle it**: a dataset whose readers are kept apart *and* whose photographs
  overlap another dataset's, so that the same photograph carries two independent committees'
  verdicts. Nothing catalogued here does that yet.

